In [ ]:
import sys
sys.path.append('..')

from nodes.summarizer import summarize
from config.schemas import Article, Summary

print('Setup complete.')

## Test 1: High-relevance article → should be included
A major model release with concrete facts. Expected: 1 summary returned for the company.

In [ ]:
high_relevance_article = Article(
    url='https://example.com/gpt5-release',
    title='OpenAI releases GPT-5',
    company='OpenAI',
    published_date='2026-03-20',
    raw_text='''
    OpenAI officially released GPT-5 on March 20, 2026.
    The model features a 2 trillion parameter architecture and achieves 95.3% on the MMLU benchmark.
    CEO Sam Altman announced the model supports a 256K context window.
    API access starts at $0.01 per 1K tokens. The model is available immediately.
    '''
)

state = {'raw_articles': [high_relevance_article]}
result = await summarize(state)

assert 'summaries' in result, 'Result must contain summaries key'
assert len(result['summaries']) == 1, f'Expected 1 summary, got {len(result["summaries"])}'

summary = result['summaries'][0]
print(f'company      : {summary.company}')
print(f'summary_text : {summary.summary_text}')
print(f'key_points   : {summary.key_points}')
print('PASSED')

## Test 2: Low-relevance article → should be filtered out
An old SEO listicle with no concrete news. Expected: empty summaries list.

In [ ]:
low_relevance_article = Article(
    url='https://example.com/top10-ai-tools',
    title='Top 10 AI Tools You Need in 2024',
    company='Industry',
    published_date='2024-01-01',
    raw_text='''
    Are you looking for the best AI tools? Here are our top picks for 2024!
    Number 1: ChatGPT. Number 2: Midjourney. Number 3: GitHub Copilot.
    Subscribe to our newsletter for more great tips and tricks!
    Click here to sign up. Privacy Policy. Terms of Service.
    '''
)

state = {'raw_articles': [low_relevance_article]}
result = await summarize(state)

assert 'summaries' in result, 'Result must contain summaries key'
assert len(result['summaries']) == 0, f'Expected 0 summaries (filtered), got {len(result["summaries"])}'

print('Low-relevance article correctly filtered out.')
print('PASSED')

## Test 3: Empty raw_text → should be skipped without API call
Simulates a scraper failure. Expected: empty summaries list, no API call made.

In [ ]:
empty_article = Article(
    url='https://example.com/blocked-article',
    title='Article behind paywall',
    company='Anthropic',
    published_date='2026-03-25',
    raw_text=''  # scraper returned nothing
)

state = {'raw_articles': [empty_article]}
result = await summarize(state)

assert len(result['summaries']) == 0, f'Expected 0 summaries, got {len(result["summaries"])}'

print('Empty article correctly skipped.')
print('PASSED')

## Test 4: Mixed batch → only relevant articles pass through
3 articles: 1 high-relevance, 1 low-relevance, 1 empty. Expected: exactly 1 summary.

In [ ]:
articles = [
    Article(
        url='https://example.com/anthropic-funding',
        title='Anthropic raises $2B Series D',
        company='Anthropic',
        published_date='2026-03-24',
        raw_text='''
        Anthropic announced a $2 billion Series D funding round on March 24, 2026.
        The round was led by Google with participation from Spark Capital.
        CEO Dario Amodei stated the funds will accelerate Claude 4 development.
        The new valuation places Anthropic at $18 billion.
        '''
    ),
    Article(
        url='https://example.com/old-news',
        title='AI Trends from Last Year',
        company='Industry',
        published_date='2025-01-01',
        raw_text='Here is a roundup of AI trends from last year. Sign up for more updates.'
    ),
    Article(
        url='https://example.com/paywall',
        title='Exclusive: Inside OpenAI',
        company='OpenAI',
        published_date='2026-03-25',
        raw_text=''  # scraper failed
    ),
]

state = {'raw_articles': articles}
result = await summarize(state)

print(f'Articles in: {len(articles)}')
print(f'Summaries out: {len(result["summaries"])}')

for s in result['summaries']:
    print(f'  - {s.company} ({len(s.articles)} articles)')

assert len(result['summaries']) == 1, f'Expected 1 summary, got {len(result["summaries"])}'
print('PASSED')

## Test 5: Output structure validation
Verify the Summary object has all required fields with correct types.

In [ ]:
article = Article(
    url='https://example.com/gemini-2',
    title='Google releases Gemini 2.0 Ultra',
    company='Google DeepMind',
    published_date='2026-03-22',
    raw_text='''
    Google DeepMind released Gemini 2.0 Ultra on March 22, 2026.
    The model achieves state-of-the-art results on HumanEval (92.1%) and MATH (88.4%).
    VP Demis Hassabis announced native multimodal support including audio and video.
    The model is available via Google AI Studio and Vertex AI starting today.
    '''
)

state = {'raw_articles': [article]}
result = await summarize(state)

assert len(result['summaries']) == 1
s = result['summaries'][0]

assert isinstance(s, Summary),           f'Expected Summary, got {type(s)}'
assert isinstance(s.company, str),        'company must be a string'
assert len(s.company) > 0,               'company must not be empty'
assert isinstance(s.articles, list),      'articles must be a list'
assert len(s.articles) > 0,              'articles must not be empty'
assert isinstance(s.summary_text, str),   'summary_text must be a string'
assert len(s.summary_text) > 0,          'summary_text must not be empty'
assert isinstance(s.key_points, list),    'key_points must be a list'
assert len(s.key_points) > 0,            'key_points must not be empty'
assert s.articles[0].url == article.url, 'article reference must be preserved'

print(f'company         : {s.company}')
print(f'articles        : {len(s.articles)}')
print(f'summary_text    : {s.summary_text}')
print(f'key_points ({len(s.key_points)}) : {s.key_points}')
print('PASSED')

## Test 6: Integration — research → scraper → summarizer (live)
Runs the full pipeline with your real TARGET_COMPANIES config. Makes real API calls to Tavily, scrapes real articles, and summarizes them.

In [ ]:
from nodes.research import research
from nodes.scraper import scrape
from nodes.summarizer import summarize

# --- Step 1: Research ---
print('Step 1: Researching articles...')
state = research({'existing_urls': set()})
print(f'  Found {len(state["search_results"])} articles')

# --- Step 2: Scrape ---
print('Step 2: Scraping articles...')
state.update(await scrape(state))
print(f'  Scraped {len(state["raw_articles"])} articles with content')

# --- Step 3: Summarize ---
print('Step 3: Summarizing...')
state.update(await summarize(state))

# --- Results ---
print(f'\n=== RESULTS: {len(state["summaries"])} company summaries ===\n')
for s in state['summaries']:
    print(f'{s.company} ({len(s.articles)} articles)')
    print(f'  Summary   : {s.summary_text[:120]}...')
    print(f'  Key points: {len(s.key_points)} points')
    print()